# 01 · export v2 production log — canonical log + log features

**Kernel: `fttl-v2` (env-v2).** v2 repo의 `lvanalytics/evaluation_helpers.py`로 프로덕션 로그를 읽고,
v3 extract의 `Fttl`을 붙여 outcome을 만든 뒤 parquet으로 내보낸다. 로그 프레임은 v2 쪽에 묶여
있으므로 여기(env-v2)서 parquet으로 떨어뜨려야 분석 env가 읽는다.

- score = 로그의 **`FastTrackerProbablity`** (repo 철자 그대로)
- observed = v3 extract의 **`Fttl`** — 로그 자체에는 outcome이 없다

| writes | kind | 컬럼 |
|---|---|---|
| `logs/v2.parquet` | `log` | `claim_id, date, score, decision, observed` |
| `logs/features_v2_log.parquet` | `log_features` | `claim_id` + MODEL_FEATURES |

경로는 둘 다 `config.path(kind, "v2", "real")`에서 나온다 — 파일명을 노트북에 박지 않는다.
`logs/v2.parquet`은 지금 **더미**가 들어 있고, 이 노트북이 그 자리를 진짜 로그로 덮어쓴다.
이 canonical log 는 `loaders.load("v2").log` 가 읽는 형식이다.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import config                      # noqa: E402

# 커널 확인: 분석용 .venv가 아니라 env-v2여야 한다 (두 줄이 같은 인터프리터인지 눈으로 확인)
print("kernel :", sys.executable)
print("env-v2 :", config.python_bin("v2"))


In [ ]:
# lvanalytics/ 를 담고 있는 디렉터리만 sys.path 에 넣으면 평범한 import 문이 통한다.
# repo 루트 바로 아래가 아니면 여기만 실제 부모로 바꾼다 (예: config.repo("v2") / "src").
sys.path.insert(0, str(config.repo("v2")))

from lvanalytics import evaluation_helpers as eh   # noqa: E402

print(eh.__file__)
print([n for n in dir(eh) if not n.startswith("_")])


---
## join → parquet

extract 에서 가져오는 것은 `Fttl` 하나뿐이다. 키 컬럼은 extract 쪽 이름(`Claimnumber_CLAIM`)을
v2 로그 이름으로 바꿔서 붙인다 — 값 형식은 양쪽이 같다.


In [ ]:
# ---- SOURCE: 두 프레임과 실제 컬럼 이름 (둘 다 polars) ------------------------------
import polars as pl                        # noqa: E402

LOG = v2_logs                              # eh 로 읽은 프로덕션 로그
EXTRACT = extract                          # v3 extract (Fttl 을 가져올 곳)

LOG_ID = "ClaimNumber"                     # 로그 쪽 claim number — 실제 이름으로 고칠 것
EXT_ID = "Claimnumber_CLAIM"               # extract 쪽
OBSERVED = "Fttl"                          # extract 쪽 라벨 = config.column('v3','observed')

print(LOG.shape, EXTRACT.shape)
print(LOG.columns)


In [ ]:
print([c for c in LOG.columns if "decis" in c.lower()])

In [ ]:
# ---- join: extract 에서 Fttl 만 붙인다 ------------------------------------------------
right = (EXTRACT.select([pl.col(EXT_ID).alias(LOG_ID), pl.col(OBSERVED)])
                .unique(subset=[LOG_ID], keep="first"))   # 중복이 있으면 join 이 행을 늘린다

joined = LOG.join(right, on=LOG_ID, how="left")           # inner 면 매칭 안 된 로그 행이 사라진다
assert joined.height == LOG.height, (joined.height, LOG.height)
print(f"join coverage {joined[OBSERVED].is_not_null().mean():.1%}"
      f"  ({joined[OBSERVED].is_not_null().sum()} / {joined.height})")


In [ ]:
# ---- MODEL_FEATURES vs 로그 컬럼 ----------------------------------------------------
# 로그가 "transformed feature" 를 담고 있다지만, 학습이 실제로 쓴 컬럼 집합과 같은지는 별개다.
# v2 repo 가 이미 sys.path 에 있으므로 param.py 를 그대로 import 해서 대조한다 (붙여넣기보다
# 안전하다 — 붙여넣은 리스트는 repo 가 바뀌면 조용히 낡는다).
try:
    from param import MODEL_FEATURES      # noqa: E402
    SRC = "param.py (import)"
except ImportError as exc:
    print("param.py import 실패:", exc, "— 아래 리스트에 param.py 내용을 그대로 붙여넣을 것")
    MODEL_FEATURES = [
        # <<< paste v2 param.py MODEL_FEATURES here, in its original order >>>
    ]
    SRC = "pasted"

assert MODEL_FEATURES, "MODEL_FEATURES 가 비어 있다"
dupes = sorted({c for c in MODEL_FEATURES if MODEL_FEATURES.count(c) > 1})
assert not dupes, f"param 리스트에 중복: {dupes}"

log_cols = list(LOG.columns)
in_log = [c for c in MODEL_FEATURES if c in log_cols]        # fit 순서 그대로 유지
missing = [c for c in MODEL_FEATURES if c not in log_cols]   # 모델이 쓰는데 로그에 없음
extra = [c for c in log_cols if c not in set(MODEL_FEATURES)]  # 로그에만 있음 (id/score/date 등)

print(f"MODEL_FEATURES {len(MODEL_FEATURES)}  ({SRC})   ·   log {len(log_cols)} columns")
print(f"  둘 다 있음   {len(in_log)}")
print(f"  로그에 없음  {len(missing)}  {missing[:20]}")
print(f"  로그에만 있음 {len(extra)}  {extra[:20]}")
print("  순서 동일   :", in_log == [c for c in log_cols if c in set(MODEL_FEATURES)])


In [ ]:
# ---- canonical 이름으로 -> log / log_features ---------------------------------------
import config                               # noqa: E402
import schema                               # noqa: E402

# 실제 컬럼명은 전부 config 에서 온다. 아직 안 채워진 것이 있으면 여기서 무엇을 채울지
# 이름을 대며 멈춘다 — 노트북에 임시로 적어두고 넘어가지 말 것.
DATE = config.column("v2", "date")
SCORE = config.column("v2", "score")           # "FastTrackerProbablity"
DECISION = config.column("v2", "decision")     # 로그의 fast-track 플래그
# decision 이름은 아직 로그로 확인된 적이 없다 — 없으면 후보를 찍고 멈춘다.
assert DECISION in joined.columns, [c for c in joined.columns
                                    if "track" in c.lower() or "decis" in c.lower()]

# feature 는 모델이 실제로 본 컬럼만, fit 순서대로.
feat_cols = [c for c in MODEL_FEATURES if c in joined.columns]
features = joined.select([pl.col(LOG_ID).alias(schema.CLAIM_ID), *feat_cols])

log = joined.select([
    pl.col(LOG_ID).alias(schema.CLAIM_ID),
    pl.col(DATE).alias(schema.DATE),
    pl.col(SCORE).alias(schema.SCORE),
    # pl.col(SCORE).alias("model_v2_score"). 
    pl.col(DECISION).alias(schema.DECISION),
    pl.col(OBSERVED).alias(schema.OBSERVED),
])

assert log[schema.CLAIM_ID].is_unique().all(), "로그에 중복 claim number"
print("log", log.shape, "· features", features.shape)
print("scrap rate", f"{log[schema.DECISION].mean():.4f}",
      "· observed pos rate", f"{log[schema.OBSERVED].mean():.4f} (null 제외)")


In [ ]:
# ---- write: 경로는 config 의 kind 에서 -----------------------------------------------
WRITTEN = []
for df, kind in ((log, "log"), (features, "log_features")):
    dst = config.path(kind, "v2", "real")
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        print(f"OVERWRITING {dst.name} — 지금까지 들어 있던 것은 make_dummy_real.py 의 더미")
    df.write_parquet(dst)
    WRITTEN.append((dst, df.height, df.columns))
    print(f"{kind:<13} -> {dst.relative_to(config.ROOT)}  rows {df.height}  cols {df.width}")


메모:

- 두 산출물은 **kind** 로 잡혀 있다: `log` 는 원래 있던 kind(`loaders.load("v2").log`,
  `log_features` 는 이 노트북 때문에 새로 추가한 kind다.
  파일명·디렉터리는 `config.FALLBACK` 한 줄에만 있고 노트북에는 없다.
- `log_features` 는 **split kind 가 아니다.** 학습 split 의 `features_v2_{split}.parquet` 은 Z:
  transformed 프레임에서 나온 행이고, 이쪽은 프로덕션이 실제로 스코어한 행이다. 다른 행 집합이므로
  섞지 말 것.
- join 은 left 로 둔다. inner 면 extract 에 없는 로그 행이 사라져 로그 행 수가 바뀐다.
- `observed` 는 v3 extract 의 `Fttl` 이고, extract 창(2023-06~2026-05) 밖의 로그 행은 null 이다.
  이 null 은 "라벨 미관측"이지 0 이 아니다 — 절대 fillna(0) 하지 말 것.
- `decision` 은 로그가 들고 있는 **실제 fast-track 플래그**여야 한다. `score >= τ` 로 만들어
  쓰지 말 것 — v2 의 τ 는 기간에 따라 다섯 번 바뀌었으므로 단일 τ 로 재구성하면 틀린다.
- `MODEL_FEATURES` 대조에서 "로그에 없음"이 하나라도 있으면 그 feature 는 파일에서 빠진 채로
  나간다. 오타인지, 로그가 실제로 안 싣는 컬럼인지 먼저 판정할 것.
